<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:31px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 12 · Understanding Regression Errors<br>and Model Complexity</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 24 November 2026 &nbsp; | &nbsp; Part 3: Supervised Learning</div>
<div style="margin-top:18px;padding-top:13px;border-top:1px solid #47667D;color:#D5F6F2;font-size:15px;">L11 taught us how to fit a prediction model. L12 asks how well its predictions work for houses it has not learned from.</div>
</div>

### The question that drives today's lecture

> **Our L11 pipeline can predict a house price. How do we know if the prediction is useful, whether the model generalizes, and whether making it more flexible would help?**

By the end, you should be able to:

- Interpret **residuals**, **Mean Absolute Error**, **Mean Squared Error**, **Root Mean Squared Error**, and the **coefficient of determination ($R^2$)**.
- Reconstruct a **fitting / validation / test** workflow without changing the original L11 test observations.
- Distinguish **fitting quality** from **generalization**; recognize signs of underfitting and overfitting.
- Explain polynomial regression and why additional flexibility does **not** guarantee better predictions.
- Select a model using validation evidence while protecting the final test set.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Return to our L11 pipeline</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">We have already built a model. What should we do with its predictions?</div>
</div>

### Start exactly where L11 stopped

Our **multifeature linear regression pipeline** predicts `SalePrice` (USD) from:

- **Numerical:** `GrLivArea` (living area), `OverallQual` (overall quality), `YearBuilt`, `LotFrontage` (may be missing).
- **Categorical:** `Neighborhood` (text categories).

Remember the L11 workflow:

**Numerical values → median imputation → standardization**

**Neighborhood → one-hot encoding**

**Combined features → linear regression → predicted house price**

The pipeline learns preprocessing statistics **and** regression coefficients during `fit()`. It can then transform a new raw row and predict with `predict()`.

**Important:** The 20% test set from L11 is deliberately untouched. Our first demonstration below uses *training observations only*—it recalls the model, but is **not yet a trustworthy estimate of performance on new houses**.

In [ ]:
# One setup cell for the entire lecture (same dataset and visual conventions as L11).
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

NAVY, TEAL, GOLD, BLUE = "#102A43", "#18B7AC", "#DB9137", "#557AB5"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10.5,
                     "axes.spines.top": False, "axes.spines.right": False})

DATA_PATH = Path("data") / "ames_housing_l11.csv"
assert DATA_PATH.is_file(), "Extract the L12 ZIP and keep the data/ folder next to this notebook."
houses = pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS = ["Id", "GrLivArea", "OverallQual", "YearBuilt",
                    "LotFrontage", "Neighborhood", "SalePrice"]
assert houses.shape == (2930, 7) and houses.columns.tolist() == EXPECTED_COLUMNS
assert houses["Id"].is_unique and houses["SalePrice"].notna().all()
FEATURES = ["GrLivArea", "OverallQual", "YearBuilt", "LotFrontage", "Neighborhood"]
X, y = houses[FEATURES].copy(), houses["SalePrice"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42  # Exact L11 split; NEVER re-randomize test.
)
assert (len(X_train), len(X_test)) == (2344, 586)
print(f"Ames sales: {len(houses):,}; L11 training rows: {len(X_train):,}; protected test rows: {len(X_test):,}")

### Rebuild the *same* L11 pipeline recipe

We are not learning a new regression algorithm yet. This is the familiar preparation-plus-prediction pipeline; encapsulating the recipe in a small function lets us instantiate a fresh unfitted pipeline later when we introduce validation.

In [ ]:
NUMERIC_FEATURES = ["GrLivArea", "OverallQual", "YearBuilt", "LotFrontage"]
CATEGORICAL_FEATURES = ["Neighborhood"]

def make_house_pipeline():
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("numeric", numeric_steps, NUMERIC_FEATURES),
        ("neighborhood", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("prepare", prepare), ("model", LinearRegression())])

# L11 recap only: this fitted estimator knows ALL of the original 80% training portion.
# We will NOT reuse this particular fitted estimator for validation later.
l11_recap_model = make_house_pipeline().fit(X_train, y_train)
preview_rows = X_train.head(5)
preview = pd.DataFrame({
    "Actual price (USD)": y_train.loc[preview_rows.index].to_numpy(),
    "Predicted price (USD)": np.round(l11_recap_model.predict(preview_rows)).astype(int),
}, index=preview_rows.index)
preview["Residual (USD)"] = preview["Actual price (USD)"] - preview["Predicted price (USD)"]
display(preview.style.format("${:,.0f}"))

### We have predictions—now we have questions

For observation $i$:

$$\boxed{e_i = y_i - \hat{y}_i}$$

- $y_i$ is the **actual recorded sale price**.
- $\hat{y}_i$ (read *y-hat*) is the **predicted sale price**.
- $e_i$ is the **residual**: the difference between actual and predicted.

A **positive residual** means the model predicted too low. A **negative residual** means it predicted too high.

We already used squared residuals in L11 to **fit coefficients**. But now we must judge the quality of predictions: 

**How do we summarize many residuals? Which error sizes should concern us? And can training predictions tell us what happens for unseen houses?**

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Measuring prediction errors</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Why do we need more than one number to describe mistakes?</div>
</div>

### A tiny example before any evaluation library

Use four **constructed** house predictions (prices in **thousands of USD**, written as **$k**). Keeping the values small makes the mathematics transparent; we return to real Ames prices shortly.

| House | Actual price (\$k) | Predicted price (\$k) | Residual (\$k) | Absolute error (\$k) | Squared error ((\$k)²) |
|---:|---:|---:|---:|---:|---:|
| 1 | 100 | 110 | −10 | 10 | 100 |
| 2 | 150 | 140 | +10 | 10 | 100 |
| 3 | 200 | 215 | −15 | 15 | 225 |
| 4 | 250 | 230 | +20 | 20 | 400 |
| **Total** | | | **+5** | **55** | **825** |

**Why not average the signed residuals?** Positive and negative mistakes partly cancel: $(−10+10−15+20)/4 = 1.25$ ($k). A small signed mean could hide large mistakes in both directions.

In [ ]:
# See the ingredients of our four worked metrics (no model fitting here).
y_true_small = np.array([100., 150., 200., 250.])     # thousand USD
y_pred_small = np.array([110., 140., 215., 230.])
residual_small = y_true_small - y_pred_small
small_table = pd.DataFrame({
    "Actual ($k)": y_true_small,
    "Predicted ($k)": y_pred_small,
    "Residual ($k)": residual_small,
    "Absolute error ($k)": np.abs(residual_small),
    "Squared error (($k)^2)": residual_small ** 2,
})
display(small_table)
print("Sum of absolute errors:", np.abs(residual_small).sum(), "thousand USD")
print("Sum of squared errors:", (residual_small**2).sum(), "(thousand USD)^2")

### Mean Absolute Error (MAE): *How wrong are we on average?*

**Meaning first.** We want an average that does not allow positive and negative residuals to cancel. Take the *absolute value* of every error, then average.

$$\boxed{\mathrm{Mean\ Absolute\ Error\ (MAE)}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|}$$

Here $n$ is the number of evaluated houses, $y_i$ their actual prices, and $\hat y_i$ their predictions.

**Work through our table:**

$$MAE = \frac{10+10+15+20}{4}=13.75\;\text{thousand USD}$$

Our predictions miss by **$13,750 in absolute terms on average**. MAE has the **same units as the target** (USD). This makes it accessible to a decision-maker.

**Limitation:** MAE treats errors proportionally: a $20k error contributes twice as much as a $10k error. It does not deliberately give extra weight to a rare very large mistake.

In [ ]:
# Calculate by hand using NumPy, then check with scikit-learn.
mae_manual = np.abs(y_true_small - y_pred_small).mean()
mae_sklearn = mean_absolute_error(y_true_small, y_pred_small)
assert np.isclose(mae_manual, 13.75) and np.isclose(mae_manual, mae_sklearn)
print(f"Mean Absolute Error (MAE): {mae_manual:.2f} thousand USD = ${mae_manual*1000:,.0f}")

### Mean Squared Error (MSE): *What if we care especially about large mistakes?*

**Meaning first.** Before averaging, *square* every residual. This makes all contributions nonnegative, and larger residuals grow disproportionately.

$$\boxed{\mathrm{Mean\ Squared\ Error\ (MSE)}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

**Work through our table:**

$$MSE=\frac{100+100+225+400}{4}=206.25\;\text{(thousand USD)}^2$$

Compare $10^2=100$ and $20^2=400$: doubling the error **quadruples** the squared penalty.

**Connection to L11:** Ordinary Least Squares minimizes the **sum** of squared training residuals. MSE divides that sum by $n$, so for a fixed set of $n$ observations they favor the same fitted coefficients. Here MSE is also an **evaluation measure** calculated on a specified set of observations.

**Important limitation:** MSE is in **squared price units**—not dollars and **never a percentage of prediction accuracy**. Its magnitude is difficult to interpret directly without context.

In [ ]:
mse_manual = np.mean((y_true_small - y_pred_small)**2)
mse_sklearn = mean_squared_error(y_true_small, y_pred_small)
assert np.isclose(mse_manual, 206.25) and np.isclose(mse_manual, mse_sklearn)
print(f"Mean Squared Error (MSE): {mse_manual:.2f} (thousand USD)^2")

### Root Mean Squared Error (RMSE): *Can we bring the result back to dollars?*

MSE emphasizes bigger mistakes, but reports squared units. We can take the square root to restore the original units:

$$\boxed{\mathrm{Root\ Mean\ Squared\ Error\ (RMSE)}=\sqrt{\mathrm{MSE}}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}}$$

**Work through our table:**

$$RMSE = \sqrt{206.25} \approx 14.36\;\text{thousand USD}$$

This is about **$14,361 in error magnitude**, expressed in price units and still more sensitive to large mistakes than MAE. **Do not interpret RMSE as the ordinary average absolute error:** MAE already has that role.

**Limitation:** One unusually bad prediction can noticeably increase RMSE. Whether this is desirable depends on the real cost of large errors.

In [ ]:
# Compatible across scikit-learn versions: square root of MSE.
rmse_manual = np.sqrt(mse_manual)
rmse_sklearn = np.sqrt(mean_squared_error(y_true_small, y_pred_small))
assert np.isclose(rmse_manual, rmse_sklearn) and np.isclose(rmse_manual, np.sqrt(206.25))
print(f"Root Mean Squared Error (RMSE): {rmse_manual:.2f} thousand USD")
print(f"In dollars: ${rmse_manual*1000:,.0f}")

### Why can MAE and RMSE tell different stories?

Compare an illustrative set of prediction errors with another set containing **one much larger error**. Their exact values are hypothetical; the point is the *sensitivity* of the metrics, not an Ames Housing claim.

In [ ]:
# Illustrative ERROR MAGNITUDES only, in $k (not another real-world dataset).
typical_errors = np.array([10., 10., 15., 20.])
one_large_error = np.array([10., 10., 15., 80.])
error_cases = [("Four moderate errors", typical_errors),
               ("One unusually large error", one_large_error)]
rows = []
for label, errors in error_cases:
    rows.append({"Case": label, "MAE ($k)": errors.mean(),
                 "RMSE ($k)": np.sqrt(np.mean(errors**2))})
error_comparison = pd.DataFrame(rows)
display(error_comparison.round(2))
fig, ax = plt.subplots(figsize=(8.7, 3.8))
positions = np.arange(len(error_comparison)); width = 0.32
ax.bar(positions-width/2, error_comparison["MAE ($k)"], width, color=TEAL,
       label="Mean Absolute Error")
ax.bar(positions+width/2, error_comparison["RMSE ($k)"], width, color=NAVY,
       label="Root Mean Squared Error")
ax.set_xticks(positions, error_comparison["Case"])
ax.set(ylabel="Error magnitude (thousand USD)", title="One large mistake has more influence on RMSE")
ax.legend(frameon=False); ax.grid(axis="y", alpha=.18)
plt.show()

**Interpretation:** Both measures rise when a prediction becomes much worse, but RMSE rises more strongly. In housing, we may care especially about very large pricing mistakes. For other applications, MAE may align more directly with the decision being made. Neither is universally superior.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">A baseline for understanding $R^2$</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">How useful is a model compared with predicting the same value for every house?</div>
</div>

### Coefficient of Determination ($R^2$): *Better than a constant-mean reference?*

Imagine that we refuse to use house features at all and predict **the mean actual price of the evaluation observations** for each house. How much squared prediction error would that produce?

For our four constructed houses, the mean is:

$$\bar y=\frac{100+150+200+250}{4}=175\;\text{thousand USD}$$

The squared-error total from this constant-mean reference is:

$$SST=(100-175)^2+(150-175)^2+(200-175)^2+(250-175)^2=12{,}500$$

Our candidate model's squared-error total from the previous table is:

$$SSE=100+100+225+400=825$$

We compare these two quantities:

$$\boxed{\text{Coefficient of Determination}\quad R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}=1-\frac{SSE}{SST}}$$

$$R^2=1-\frac{825}{12{,}500}=0.934$$

**Meaning:** Compared with the evaluation-set mean reference, our model reduced the total squared error substantially. It is **not correct** to call this *93.4% accurate predictions*, and $R^2$ is **not generally the square of a correlation coefficient**.

In [ ]:
reference_mean = y_true_small.mean()
model_sse = np.sum((y_true_small - y_pred_small)**2)
reference_sst = np.sum((y_true_small - reference_mean)**2)
r2_manual = 1 - model_sse/reference_sst
r2_sklearn = r2_score(y_true_small, y_pred_small)
assert np.isclose(reference_mean, 175) and np.isclose(reference_sst, 12500)
assert np.isclose(r2_manual, 0.934) and np.isclose(r2_manual, r2_sklearn)
print(f"Evaluation-set mean = {reference_mean:.0f} ($k)")
print(f"Model squared error SSE = {model_sse:.0f}; mean-reference SST = {reference_sst:.0f}")
print(f"Coefficient of Determination (R²) = {r2_sklearn:.3f}")

### What values can $R^2$ take?

- **$R^2=1$:** perfect predictions.
- **$R^2=0$:** squared error equal to the constant prediction of the **evaluation-set mean**.
- **$R^2<0$:** more squared error than that reference, possible on unseen observations.
- **High $R^2$:** useful evidence in context, but does **not** guarantee small dollar errors or justify every use of the model.

**A subtle but important distinction:** `DummyRegressor(strategy="mean")` learns a mean from its **fitting data**, whereas the mathematical $R^2$ denominator uses the mean of **the evaluation targets**. On a separate validation set, those two means can differ. Therefore, a trained dummy mean predictor need not receive exactly $R^2=0$ on validation.

In [ ]:
# An intentionally poor constructed prediction can produce a negative R².
bad_predictions = np.array([250., 250., 250., 250.])
print(f"Poor illustrative prediction R²: {r2_score(y_true_small, bad_predictions):.3f}")
assert r2_score(y_true_small, bad_predictions) < 0

### Which metric should we report?

| Full name | Abbreviation | What it conveys | Units | Main caution |
|---|---|---|---|---|
| **Mean Absolute Error** | **MAE** | Average absolute distance from actual prices | USD | Does not specially emphasize very large errors |
| **Mean Squared Error** | **MSE** | Average squared penalty | USD² | Hard to interpret directly; sensitive to large errors |
| **Root Mean Squared Error** | **RMSE** | Error magnitude in price units; sensitive to large errors | USD | Strongly influenced by outliers |
| **Coefficient of Determination** | **$R^2$** | Squared error relative to a mean reference | Unitless | Not classification accuracy; can be negative |

**For today's comparison:** we will use **validation RMSE** as the principal comparison number, while also interpreting **MAE** and **$R^2$**. We will never interpret a prediction error as a "percentage of accurate houses".

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">From training data to reliable validation</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">What evidence do we need before trusting our predictions?</div>
</div>

### Why can't we simply report errors on L11's training observations?

A model was fitted to reduce its **training residuals**. If we assess the same rows, we are checking how well it describes observations it has already used—not how well it predicts new ones.

For L12, each partition has a different purpose:

| Partition | Purpose | How it may be used |
|---|---|---|
| **Fitting (60%)** | Learn imputation, scaling, encoding, regression coefficients | `fit()` and measure training error |
| **Validation (20%)** | Compare candidate models, choose polynomial degree | `predict()`, calculate metrics, choose |
| **Test (20%)** | One final estimate after selection decisions are fixed | **Reserved for Assignment 06; not evaluated in this lecture** |

**We must preserve the original L11 test rows.** First reproduce L11's 80%/20% split; then split **only its 80% training portion** into fitting and validation (75%/25% of that training portion). The resulting proportions are 60% / 20% / 20% of all houses.

Because the L11 recap model was fitted on *all* original training rows, it has already seen the future validation rows. **We cannot score that fitted estimator as if its validation observations were new.** We create a fresh pipeline using the same recipe, fitted only on the new fitting set.

In [ ]:
# Validation is drawn only from the 80% partition created by the original L11 split.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)
assert (len(X_fit), len(X_val), len(X_test)) == (1758, 586, 586)
assert X_fit.index.intersection(X_val.index).empty
assert X_fit.index.intersection(X_test.index).empty
assert X_val.index.intersection(X_test.index).empty
assert set(X_fit.index).union(X_val.index) == set(X_train.index)
assert set(X_train.index).isdisjoint(X_test.index)
print(f"Fitting: {len(X_fit):,} | Validation: {len(X_val):,} | Protected test: {len(X_test):,}")

fig, ax = plt.subplots(figsize=(8.7, 1.8))
ax.barh([0], [len(X_fit)], color=TEAL, label=f"Fit {len(X_fit):,}")
ax.barh([0], [len(X_val)], left=[len(X_fit)], color=BLUE, label=f"Validation {len(X_val):,}")
ax.barh([0], [len(X_test)], left=[len(X_fit)+len(X_val)], color=GOLD, label=f"Protected test {len(X_test):,}")
ax.set(xlim=(0, len(houses)), yticks=[], xlabel="Number of Ames observations",
       title="L12: Split the original L11 training portion, not the original test set")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.35), ncol=3, frameon=False)
plt.show()

### Create the properly fitted L12 model

The pipeline structure is unchanged. What changes is **which observations it is allowed to learn from**.

- `SimpleImputer` learns numerical medians from `X_fit`.
- `StandardScaler` learns means and scales from `X_fit`.
- `OneHotEncoder` learns categories from `X_fit`.
- `LinearRegression` learns coefficients from `(X_fit, y_fit)`.
- `X_val` is transformed and predicted, **never fitted**.

In [ ]:
# Fresh, unfitted copy of the familiar L11 modeling recipe.
full_linear_model = make_house_pipeline()
full_linear_model.fit(X_fit, y_fit)

numeric_imputer = (full_linear_model.named_steps["prepare"]
                   .named_transformers_["numeric"].named_steps["impute"])
fit_medians = pd.Series(numeric_imputer.statistics_, index=NUMERIC_FEATURES,
                        name="Median learned from fitting observations")
display(fit_medians.to_frame())
print("Fitted pipeline steps:", list(full_linear_model.named_steps))
print("Validation rows never entered fit().")

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Evaluate real Ames predictions</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Our first honest answer: how well does the L11 pipeline generalize to held-out houses?</div>
</div>

### One model, two distinct sets of predictions

We can now evaluate the **same fitted model** on the fitting and validation sets. Comparing those results is more informative than evaluating only on observations that the model learned from.

We already learned the arithmetic of each metric, so a small reporting function is now appropriate. It will calculate the same values directly from actual and predicted prices.

In [ ]:
# This helper is introduced AFTER the manual calculations, not instead of them.
def regression_scores(actual, predicted):
    return {
        "MAE (USD)": mean_absolute_error(actual, predicted),
        "MSE (USD^2)": mean_squared_error(actual, predicted),
        "RMSE (USD)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }

fit_predictions = full_linear_model.predict(X_fit)
val_predictions = full_linear_model.predict(X_val)
results_full = pd.DataFrame({
    "Fitting": regression_scores(y_fit, fit_predictions),
    "Validation": regression_scores(y_val, val_predictions),
}).T
display(results_full.style.format({
    "MAE (USD)": "${:,.0f}", "MSE (USD^2)": "{:,.0f}",
    "RMSE (USD)": "${:,.0f}", "R²": "{:.3f}"}))

**Interpretation questions:** On validation, about how many dollars does the model miss by according to MAE? What does RMSE add? What does its coefficient of determination say relative to the validation mean reference?

Do not conclude that a smaller *fitting* score guarantees a smaller validation error. The two sets may differ in difficulty, extreme values or target distribution. A validation score is evidence about predictive generalization—not a guarantee for every future house.

In [ ]:
# Actual-vs-predicted values for VALIDATION houses only.
fig, ax = plt.subplots(figsize=(7.7, 5.4))
ax.scatter(y_val/1000, val_predictions/1000, color=TEAL,
           alpha=.48, s=24, linewidth=0, label="Validation houses")
limits = [min(y_val.min(), val_predictions.min())/1000,
          max(y_val.max(), val_predictions.max())/1000]
ax.plot(limits, limits, color=NAVY, linestyle="--", lw=2,
        label="Perfect prediction (actual = predicted)")
ax.set(xlim=limits, ylim=limits, xlabel="Actual price (thousand USD)",
       ylabel="Predicted price (thousand USD)",
       title="L11 multifeature pipeline: validation predictions")
ax.legend(frameon=False); ax.grid(alpha=.15)
plt.show()

**Reading the plot:** A dot on the dashed diagonal represents a perfect prediction. The vertical distance from the diagonal is its prediction error. Pay attention to houses far from the diagonal; the plot does not tell us *why* the model missed them.

The dataset contains houses with unusually high selling prices. A few large errors can have a disproportionate effect on squared-error-based metrics.

In [ ]:
# A real validation prediction table: concrete prices and signed residuals.
val_examples = pd.DataFrame({
    "Actual (USD)": y_val.to_numpy(),
    "Predicted (USD)": val_predictions,
    "Residual (USD)": y_val.to_numpy()-val_predictions,
    "Absolute error (USD)": np.abs(y_val.to_numpy()-val_predictions),
}, index=X_val.index)
# Deterministic examples chosen without changing the model or validation split.
display(val_examples.head(6).round(0).style.format({
    "Actual (USD)": "${:,.0f}", "Predicted (USD)": "${:,.0f}",
    "Residual (USD)": "${:+,.0f}", "Absolute error (USD)": "${:,.0f}"}))

### A baseline without house features

Before claiming that our feature-based model adds value, compare it with a very simple rule: **predict the mean fitting-set price for every house**. `DummyRegressor` learns that mean from `y_fit` and can predict on `X_val` without using feature values.

In [ ]:
mean_baseline = DummyRegressor(strategy="mean")
mean_baseline.fit(X_fit, y_fit)
mean_val_predictions = mean_baseline.predict(X_val)
baseline_comparison = pd.DataFrame({
    "Fitting-mean baseline": regression_scores(y_val, mean_val_predictions),
    "L11 multifeature pipeline": regression_scores(y_val, val_predictions),
}).T
display(baseline_comparison[["MAE (USD)", "RMSE (USD)", "R²"]].style.format({
    "MAE (USD)": "${:,.0f}", "RMSE (USD)": "${:,.0f}", "R²": "{:.3f}"}))
print(f"Mean price from FITTING subset: ${y_fit.mean():,.0f}")
print(f"Mean price of VALIDATION targets: ${y_val.mean():,.0f}")
print("These two means differ, so the fitted mean baseline can have a slightly nonzero validation R².")

**What evidence do we have now?** The multifeature pipeline uses genuinely useful information: its validation RMSE is substantially lower than that of a fit-mean baseline. However, we still need to ask whether the model is too simple, unnecessarily complex, or sensitive to the observations on which it was fitted.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Generalization and model complexity</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">When does fitting the training data better stop helping?</div>
</div>

### Three ideas, one problem

**Generalization:** Predicting useful values for observations that did not participate in fitting.

**Underfitting:** A model lacks enough flexibility to represent an important relationship. We may observe **large fitting and validation errors**, or a systematic pattern left unexplained.

**Overfitting:** A model follows training-specific fluctuations too closely. Often it achieves **very low fitting error but notably worse validation error**. This is a common pattern, not an unconditional mathematical rule.

A model's flexibility is its ability to represent different functional shapes. More flexibility can capture genuine patterns, but can also capture noise or unusual cases.

**Key point:** An improvement on fitting data is *not automatically* an improvement in predictive performance.

### A deliberately constructed visual intuition

The following small simulated example is **not Ames Housing and not an additional real-world dataset**. Its only purpose is to show what a simple, appropriate and overly flexible curve can look like when observations contain noise. We return immediately to Ames for the real comparison.

In [ ]:
# Conceptual-only demonstration: a known, curved trend with noisy observations.
rng = np.random.default_rng(7)
x_demo = np.linspace(-2.1, 2.1, 13)
y_demo_clean = 1.2 + 1.1*x_demo - 0.65*x_demo**2
y_demo = y_demo_clean + rng.normal(0, .32, len(x_demo))
x_draw = np.linspace(-2.2, 2.2, 200)
fig, ax = plt.subplots(figsize=(9.5, 4.7))
ax.scatter(x_demo, y_demo, s=60, color=NAVY, zorder=4, label="Constructed noisy observations")
for degree, color, label in [(1, BLUE, "Degree 1: too simple"),
                             (2, TEAL, "Degree 2: captures curvature"),
                             (10, GOLD, "Degree 10: very flexible")]:
    conceptual = Pipeline([
        ("scale", StandardScaler()),
        ("polynomial", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regression", LinearRegression()),
    ])
    conceptual.fit(x_demo.reshape(-1,1), y_demo)
    ax.plot(x_draw, conceptual.predict(x_draw.reshape(-1,1)),
            color=color, lw=2.2 if degree != 10 else 1.8,
            label=label)
ax.set(xlabel="Illustrative input x", ylabel="Illustrative target y",
       title="A constructed example: flexibility can fit noise")
ax.legend(frameon=False); ax.grid(alpha=.14)
plt.show()

**Do not generalize from this illustration alone.** The real Ames dataset is much larger. Polynomial degrees that make dramatic curves for 13 illustrative points might not cause similarly dramatic overfitting among 1,758 real fitting houses. We must measure actual validation error rather than assume the outcome.

*Bias–variance intuition:* An overly rigid model can miss genuine structure; a very flexible model can be sensitive to the particular examples it sees. 

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Can a curved function help?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Polynomial regression changes the shape of a prediction function.</div>
</div>

### From a straight line to a polynomial

In L11, our **area-only model** had the form:

$$\hat y=b_0+b_1x$$

What if increasing above-ground living area has a **curved**, rather than exactly straight-line, relationship with price? We could add a squared feature:

$$\hat y=b_0+b_1x+b_2x^2\quad\text{(quadratic, degree 2)}$$

A cubic adds $b_3x^3$, and more generally a degree-$d$ polynomial uses powers up to $x^d$.

- $x$ is `GrLivArea` (house living area); $x^2$ is a **new derived feature**.
- The $b_i$ terms are **model parameters**: coefficients learned from fitting observations.
- The polynomial **degree** is a **hyperparameter**: a configuration we choose *before* fitting.
- A polynomial curve can be nonlinear in $x$ while remaining **linear in its learned coefficients** $b_i$.

This connects directly to L08–L10: create transformed features **inside** a pipeline, then fit a familiar `LinearRegression` estimator on those features.

### What does `PolynomialFeatures` actually create?

The following illustration uses three small area values (in units of 1,000 square feet) just to show the transformation. It does not fit a model or reveal any test observations.

In [ ]:
illustrative_area = pd.DataFrame({"Area (thousand sq ft)": [1., 1.5, 2.]})
poly_example = PolynomialFeatures(degree=2, include_bias=False)
created = poly_example.fit_transform(illustrative_area)
display(pd.DataFrame(created,
    columns=["x = original area", "x² = squared area"]))
print("The intercept is learned by LinearRegression, so include_bias=False avoids a duplicate constant column.")

### A fair experiment: degree 1, 2, 3, or 5?

We now use **only `GrLivArea`**, so the comparison isolates the effect of polynomial **shape**. The L11 multifeature pipeline used many more *original facts* about a house; that is a different kind of model improvement.

Keep the same **1,758 fitting** and **586 validation** observations for every degree. Learn scaling and coefficients **only** from fitting. Because high powers of raw square-foot values can become numerically huge, scale inside the pipeline **before** generating polynomial terms.

We compare:

- **Degree 1**: straight line (the area-only L11 idea).
- **Degree 2**: one squared term.
- **Degree 3**: one cubic term.
- **Degree 5**: a more flexible curve (not automatically superior).

In [ ]:
# Define the single-feature pipeline once, then vary only the degree.
def make_area_polynomial_pipeline(degree):
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("polynomial", PolynomialFeatures(degree=degree, include_bias=False)),
        ("model", LinearRegression()),
    ])

X_fit_area = X_fit[["GrLivArea"]]
X_val_area = X_val[["GrLivArea"]]
poly_degrees = [1, 2, 3, 5]
poly_models = {}
poly_rows = []
for degree in poly_degrees:
    candidate = make_area_polynomial_pipeline(degree)
    candidate.fit(X_fit_area, y_fit)
    predicted_fit = candidate.predict(X_fit_area)
    predicted_val = candidate.predict(X_val_area)
    poly_models[degree] = candidate
    poly_rows.append({
        "Degree": degree,
        "Fitting RMSE (USD)": np.sqrt(mean_squared_error(y_fit, predicted_fit)),
        "Validation RMSE (USD)": np.sqrt(mean_squared_error(y_val, predicted_val)),
        "Validation MAE (USD)": mean_absolute_error(y_val, predicted_val),
        "Validation R²": r2_score(y_val, predicted_val),
    })
poly_results = pd.DataFrame(poly_rows)
display(poly_results.style.format({
    "Fitting RMSE (USD)": "${:,.0f}",
    "Validation RMSE (USD)": "${:,.0f}",
    "Validation MAE (USD)": "${:,.0f}",
    "Validation R²": "{:.3f}"}))

### First look at the fitted shapes

All curves below are fitted using **only the fitting observations**. For readability, the display window concentrates on common living-area values; the models were still fitted on **all** fitting observations. We should not extrapolate polynomial curves confidently to very large house sizes outside that display range.

In [ ]:
# Focus the view on the common sizes; fitting remains on the FULL fitting subset.
x_low = max(400, np.percentile(X_fit_area["GrLivArea"], 1))
x_high = np.percentile(X_fit_area["GrLivArea"], 98.5)
grid_area = np.linspace(x_low, x_high, 250)
grid_df = pd.DataFrame({"GrLivArea": grid_area})
plot_mask = X_fit_area["GrLivArea"].between(x_low, x_high)
fig, ax = plt.subplots(figsize=(10.0, 5.2))
ax.scatter(X_fit_area.loc[plot_mask, "GrLivArea"], y_fit.loc[X_fit_area.loc[plot_mask].index]/1000,
           color="#C2D1D9", s=17, alpha=.42, linewidth=0, label="Fitting observations (visible range)")
for degree, color in [(1, NAVY), (2, TEAL), (3, BLUE), (5, GOLD)]:
    ax.plot(grid_area, poly_models[degree].predict(grid_df)/1000,
            label=f"Degree {degree}", color=color, linewidth=2.25)
ax.set(xlabel="Above-ground living area (sq ft)", ylabel="Price (thousand USD)",
       title="Ames: polynomial curves change with degree")
ax.grid(alpha=.13); ax.legend(frameon=False, ncol=2)
plt.show()

**Look for:** Curves can bend, and higher degree offers extra flexibility. The shape alone does **not** tell us which candidate predicts held-out houses best. For that we need the validation comparison.

In [ ]:
# Training-vs-validation comparison: the evidence that should guide complexity decisions.
fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.plot(poly_results["Degree"], poly_results["Fitting RMSE (USD)"],
        "o-", color=NAVY, lw=2.4, ms=7, label="Fitting RMSE")
ax.plot(poly_results["Degree"], poly_results["Validation RMSE (USD)"],
        "o-", color=TEAL, lw=2.4, ms=7, label="Validation RMSE")
ax.set(xticks=poly_degrees, xlabel="Polynomial degree (chosen before fit)",
       ylabel="Root Mean Squared Error (USD)",
       title="Ames: lower fitting error does not guarantee lower validation error")
ax.grid(alpha=.18); ax.legend(frameon=False)
plt.show()

### What do the real results say?

- Among these candidates, **increasing degree reduces fitting error**. That is not evidence by itself that predictions improve for new houses.
- The validation errors are relatively close: the higher-degree candidates do **not** provide a clear validation improvement in this split.
- Degree 5 is **not** dramatic overfitting on this dataset; we should describe exactly what the measurements show, without manufacturing a story.
- A degree 1 model is a defensible choice for **this area-only experiment** if it achieves the lowest validation RMSE, especially given its simplicity.

This experiment is about *model complexity from polynomial transformation*, not about whether living area is the only useful house feature.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Choose models responsibly</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">More input information or more polynomial complexity—which actually helps?</div>
</div>

### Compare three modeling ideas fairly

Every row in this comparison uses the **same fitting and validation partition**:

1. **Area-only linear regression:** one original input, straight line (polynomial degree 1).
2. **Area-only polynomial regression:** one original input, more flexible curve.
3. **L11 multifeature linear pipeline:** more *original* housing information, plus the familiar numerical and categorical preprocessing.

Do not mistake "more features" for "higher polynomial degree". They answer different questions.

In [ ]:
# Select the polynomial degree using VALIDATION RMSE only.
selected_degree = int(poly_results.loc[poly_results["Validation RMSE (USD)"].idxmin(), "Degree"])
selected_poly = poly_models[selected_degree]
models_to_compare = {
    "Area-only degree 1": poly_models[1],
    "Area-only degree 5": poly_models[5],
}
comparison_rows = []
for label, model in models_to_compare.items():
    score = regression_scores(y_val, model.predict(X_val_area))
    comparison_rows.append({"Model": label, **score})
comparison_rows.append({
    "Model": "L11 multifeature linear pipeline",
    **regression_scores(y_val, val_predictions)
})
model_comparison = pd.DataFrame(comparison_rows).drop_duplicates(subset=["Model"])
display(model_comparison[["Model", "MAE (USD)", "RMSE (USD)", "R²"]].style.format({
    "MAE (USD)": "${:,.0f}", "RMSE (USD)": "${:,.0f}", "R²": "{:.3f}"}))
print(f"Selected area-only polynomial degree from validation: {selected_degree}")

**What do these comparisons teach?** Adding relevant original house features (quality, age, frontage, neighborhood) can be more valuable than adding high-order powers of living area.

**Parameters vs hyperparameters:** Regression coefficients are **learned during `fit()`**. Polynomial degree is a **modeling choice** made before fitting. We tested several degrees using validation evidence, not training error alone.

**One more step in a real workflow:** After fixing the configuration using validation, refit a fresh pipeline on the **entire original 80% training portion**, then evaluate **once** on the previously untouched 20% test portion. 

In [ ]:
# Final test evaluation: our model was selected using validation only.

# Rebuild and fit the selected L11 pipeline
final_pipeline = make_house_pipeline()
final_pipeline.fit(X_train, y_train)

# Predict the previously unseen test observations
test_predictions = final_pipeline.predict(X_test)

# Evaluate using our regression metrics
final_test_results = pd.DataFrame(
    [regression_scores(y_test, test_predictions)],
    index=["Final held-out test"]
)

display(final_test_results)

### Closing reflection

1. **A prediction is not an evaluation.** Compare actual and predicted values.
2. **Mean Absolute Error** is average absolute error; **Mean Squared Error** emphasizes large mistakes; **Root Mean Squared Error** puts that emphasis back into original units.
3. **Coefficient of Determination ($R^2$)** compares squared error with a mean reference, and can be negative.
4. **Fitting error** and **validation error** answer different questions.
5. **More flexible** does not automatically mean **more reliable**.
6. Choose using **validation**. Protect the **test** partition until the configuration is finalized.

> **The L11 → L12 learning shift:** We no longer ask only *Can the model fit?* We ask *What evidence suggests the model will generalize?*

### Acknowledgments and technical sources

- **Ames Housing:** Dean De Cock, *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project* (2011). The project-provided `AmesHousing.csv` contains 2,930 records; this standalone notebook bundles the same seven-field reduction used in L11. See `DATASET_README.md`.
- Lecture inspiration: Prof. Antonio Liotta's predictive analysis, polynomial regression, model validation and model tuning materials. 
- scikit-learn documentation: [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html).
- The dataset used for **Assignment 06** is a **separate Kaggle download**; it is not required for running the lecture code. Follow the dataset link and download instructions in the assignment section below.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:12px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">09 · LECTURE 12</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Assignment 06</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Independent prediction workflow: a new student-performance dataset</div>
</div>


## Assignment 06 — Student Performance: Your Regression Pipeline

### The challenge

Apply what you learned in **L10–L12** to a dataset we have **not** analyzed together. Using **Student Performance (Multiple Linear Regression)**, investigate whether you can predict the numerical **`Performance Index`** from the available student information.

### Get the dataset

Open **[Student Performance (Multiple Linear Regression) on Kaggle](https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression)**, sign in if necessary, and select **Download**. Extract the downloaded ZIP and place its CSV (usually `Student_Performance.csv`) in a `data/` folder next to your notebook. You do **not** need a Kaggle API key or an in-notebook download. Check the actual columns yourself before starting.

### Your work

Build a **complete scikit-learn preprocessing-and-regression pipeline**, including appropriate treatment of numerical and categorical inputs, and a **linear regression model**. Decide how to explore, prepare and split the data using the principles learned in class; document your choices briefly (create your own notebook with markdown to briefly state your choices followed by code cell to perform each step). 

Remember: Do not use the target as an input.

Evaluate the model using **fitting and final test observations**. Report **Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and Coefficient of Determination ($R^2$)**, and include **one informative plot**. Explain what the scores mean in the dataset's units and what they tell you about generalization.

**Optional extension:** Investigate a **polynomial regression** pipeline or an alternative feature selection or extraction. If you compare configurations, choose using **validation** results, then refit the selected configuration on the original training portion before evaluating **once** on the test set. Do not revise your choice based on test performance.

### Submit

- A new notebook: **`Assignment06.ipynb`**, which includes: your executable solution, evaluation results, a plot, and a short conclusion.
- The **original dataset CSV**, unchanged, so your notebook can run offline.

**The goal is independent investigation.** You should decide which preprocessing choices are suitable, how to organize your pipeline, and how to interpret the evidence.